---
## 1. Install Dependencies

In [0]:
%pip install unitycatalog-ai[databricks] numpy
%pip install openpyxl
%pip install thefuzz
%pip install databricks_mcp
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp nest_asyncio "gepa>=0.1.0"
%pip install backoff
%pip install databricks_openai
%pip install dspy-ai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Attempting uninstall: gepa
    Found existing installation: gepa 0.1.1
    Uninstalling gepa-0.1.1:
      Successfully uninstalled gepa-0

---
## 2. Download the Datasets
Dataset link 1: [Food Substances](https://www.hfpappexternal.fda.gov/scripts/fdcc/index.cfm?set=FoodSubstances) - FDA

Dataset link 2: [Restaurant Nutrition Datasets](https://www.menustat.org/uploads/1/4/1/6/141624194/ms_annual_data_2022.xls) - MenuStat


In [0]:
#Dataset 1
import pandas as pd
import os

# Get the directory path where the notebook is running
current_dir = os.getcwd()


# Read the FDA_Fodd_Additives.csv file into a DataFrame
df = pd.read_csv(os.path.join(current_dir, "FDA_Food_Additives.csv"))

# Optionally show the first few records
print("First 5 records:")
print(df.head())
print("\nShape:", df.shape)

First 5 records:
                              Substance  ...  JECFA Flavor Number
0                    ACAI BERRY EXTRACT  ...                  NaN
1                  ACESULFAME POTASSIUM  ...                  NaN
2                                ACETAL  ...                  941
3                          ACETALDEHYDE  ...                   80
4  ACETALDEHYDE, BUTYL PHENETHYL ACETAL  ...                 1001

[5 rows x 18 columns]

Shape: (3971, 18)


In [0]:
# Unity Catalog location: catalog.schema.table
catalog = "main"
schema = "default"
table_name = "fda_food_additives"

# Clean column names to remove invalid characters for Delta
df = df.rename(columns=lambda x: x.replace('ï»¿', '').replace(':', '_').replace(' ', '_').replace('(', '_').replace(')', '_'))

# Convert Pandas DataFrame to Spark and write as Delta
# mode("overwrite") replaces the table if it exists (good for re-running the lab)
fda_food_additives = spark.createDataFrame(df)
fda_food_additives.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

# Verify: query the table (optional)
display(spark.table(f"{catalog}.{schema}.{table_name}").limit(5))

Substance,Health_Risk_Score_1_to_10,Health_Risk_Level,Synthetic_Content_Score_1_to_10,Clean_Label_Concern_Score_1_to_10,Ultra_Processed_Association,Risk_Categories,Functional_Category,Technical_Function,Suggested_Alternatives,Allergen_or_Sensitivity_Risk,Common_Food_Uses,Regulatory_Status,CAS_Number,Regulatory_References__21_CFR_,FEMA_GRAS_Number,GRAS_Publication_Number,JECFA_Flavor_Number
"DOG GRASS, EXTRACT (AGROPYRON REPENS (L.) BEAUV.)",3,Safe,2,1,No,General_Approved,Flavoring Agent,"FLAVOR ENHANCER, FLAVORING AGENT OR ADJUVANT","Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,977038-73-5,182.2,2403,3,null
"D-8-P-MENTHENE-1,2-EPOXIDE",4,Safe,5,5,No,General_Approved,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,203719-54-4,null,4655,24,2145
"DRAGON'S BLOOD, EXTRACT (DAEMONOROPS SPP. OR OTHER BOTANICAL SOURCES)",3,Safe,2,1,No,General_Approved,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,9000-19-5,172.51,2404,3,null
DRIED ALGAE MEAL,4,Safe,5,5,No,General_Approved,Flavoring Agent,"FLAVOR ENHANCER, FLAVORING AGENT OR ADJUVANT","Natural plant extracts, essential oils, or fermentation-derived flavors",null,"Beverages, candies, baked goods, snacks, processed foods",null,977010-47-1,73.275,null,null,null
DULCIN--PROHIBITED,5,Moderate,8,9,No,General_Approved,Sweetener,NON-NUTRITIVE SWEETENER,"Stevia, Monk fruit extract, or Allulose",null,"Soft drinks, sugar-free products, baked goods, chewing gum, tabletop sweeteners",189.145,150-69-6,189.145,null,null,null


In [0]:
#Dataset 2
import pandas as pd
import requests
from io import BytesIO

url = "https://www.menustat.org/uploads/1/4/1/6/141624194/ms_annual_data_2022.xls"

# Download with headers to avoid 403 Forbidden
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(url, headers=headers)
response.raise_for_status()

# Read Excel from bytes
df = pd.read_excel(BytesIO(response.content))

# Optionally show the first few records
print("First 5 records:")
print(df.head())
print("\nShape:", df.shape)

First 5 records:
   matched_2021  new_item_2022  ...  sugar_text protein_text
0             1              0  ...         NaN          NaN
1             1              0  ...         NaN          NaN
2             1              0  ...         NaN          NaN
3             1              0  ...         NaN          NaN
4             1              0  ...         NaN          NaN

[5 rows x 33 columns]

Shape: (26238, 33)


In [0]:
# Unity Catalog location: catalog.schema.table
catalog = "main"
schema = "default"
table_name = "restaurant_nutrition_data"

# Clean column names to remove invalid characters for Delta
df = df.rename(columns=lambda x: x.replace('ï»¿', '').replace(':', '_').replace(' ', '_').replace('(', '_').replace(')', '_'))

# Convert object columns with mixed types to string to avoid Arrow conversion errors
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str)

# Convert Pandas DataFrame to Spark and write as Delta
# mode("overwrite") replaces the table if it exists (good for re-running the lab)
restaurant_nutrition_data = spark.createDataFrame(df)
restaurant_nutrition_data.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

# Verify: query the table (optional)
display(spark.table(f"{catalog}.{schema}.{table_name}").limit(5))

matched_2021,new_item_2022,menu_item_id,food_category,restaurant,item_name,item_description,serving_size,serving_size_text,serving_size_unit,serving_size_household,calories,total_fat,saturated_fat,trans_fat,cholesterol,sodium,carbohydrates,dietary_fiber,sugar,protein,potassium,notes,calories_text,total_fat_text,saturated_fat_text,trans_fat_text,cholesterol_text,sodium_text,carbohydrates_text,dietary_fiber_text,sugar_text,protein_text
1,0,939751,Sandwiches,Quiznos,Carbonara Sammie,"Carbonara Sammie, Bacon, Provolone, Sautéed Mushrooms, Parmesan Alfredo Sauce, Chicken",nan,nan,nan,nan,460,25.0,6,0,60,1100,30,3,3,26,null,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,0,939752,Sandwiches,Quiznos,Baja Sammie,"Baja Sammie, Bacon, Cheddar, Onions, BBQ Sauce, Chipotle Mayo, Chicken",nan,nan,nan,nan,410,19.0,6,0,60,1190,32,3,6,25,null,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,0,939753,Sandwiches,Quiznos,Apple Harvest Sammie,"Apple Harvest Sammie, Chicken, Apples, Pumpkin Seeds, Craisins, Tomatoes, Honey Mustard, Specialties",nan,nan,nan,nan,410,20.0,3.5,0,25,670,43,5,14,15,null,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,0,939754,Sandwiches,Quiznos,Black Angus Steakhouse Sammie,"Black Angus Steakhouse Sammie, Provolone, Cheddar, Sautéed Mushrooms & Onions, Grille Sauce on Rosemary Parmesan Bread, Steak",nan,nan,nan,nan,390,15.0,6,0,50,1000,39,4,10,23,null,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,0,939755,Sandwiches,Quiznos,Chipotle Steak & Cheddar Sammie,"Chipotle Steak & Cheddar Sammie, Sautéed Peppers & Onions, Chipotle Mayo, Steak",nan,nan,nan,nan,440,26.0,7,0,50,980,30,4,2,19,null,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


---
## 3. Create Unity Catalog Function Tools

In [0]:
%sql
--SQL Tool 1
CREATE OR REPLACE FUNCTION main.default.get_additive_info(
  additive_name STRING COMMENT 'Name or partial name of the additive to look up'
)
RETURNS TABLE(
  substance STRING COMMENT 'Full additive name',
  health_risk_score INT COMMENT 'Health risk score (1-10)',
  health_risk_level STRING COMMENT 'Risk level (Safe, Low, Moderate, High)',
  functional_category STRING COMMENT 'What the additive does',
  technical_function STRING COMMENT 'Specific technical purpose',
  common_food_uses STRING COMMENT 'Where it\'s commonly found',
  allergen_risk STRING COMMENT 'Allergen or sensitivity concerns',
  suggested_alternatives STRING COMMENT 'Safer alternatives',
  regulatory_status STRING COMMENT 'FDA regulatory status'
)
COMMENT 'Looks up detailed information about a food additive including health risks, uses, and alternatives. Supports partial name matching.'
RETURN
  SELECT 
    Substance as substance,
    Health_Risk_Score_1_to_10 as health_risk_score,
    Health_Risk_Level as health_risk_level,
    Functional_Category as functional_category,
    Technical_Function as technical_function,
    Common_Food_Uses as common_food_uses,
    Allergen_or_Sensitivity_Risk as allergen_risk,
    Suggested_Alternatives as suggested_alternatives,
    Regulatory_Status as regulatory_status
  FROM main.default.fda_food_additives
  WHERE LOWER(Substance) LIKE CONCAT('%', LOWER(additive_name), '%')
  ORDER BY Health_Risk_Score_1_to_10 DESC, Substance;

In [0]:
%sql
--Test the tool
SELECT * 
FROM main.default.get_additive_info('HEPTEN')
LIMIT 10;

substance,health_risk_score,health_risk_level,functional_category,technical_function,common_food_uses,allergen_risk,suggested_alternatives,regulatory_status
(+/-)-1-HEPTEN-3-OL,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
(E)-2-HEPTENOIC ACID,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
(Z)-4-HEPTEN-1-OL,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
"2,6-DIMETHYL-5-HEPTENAL",4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
"2,6-DIMETHYL-5-HEPTENAL PROPYLENEGLYCOL ACETAL",4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
"2,6-DIMETHYL-6-HEPTEN-1-OL",4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
2-ETHYL-2-HEPTENAL,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
2-HEPTEN-4-ONE,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
2-HEPTENAL,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null
3-HEPTEN-2-ONE,4,Safe,Flavoring Agent,FLAVORING AGENT OR ADJUVANT,"Beverages, candies, baked goods, snacks, processed foods",null,"Natural plant extracts, essential oils, or fermentation-derived flavors",null


In [0]:
%sql
--SQL Tool 2
CREATE OR REPLACE FUNCTION main.default.analyze_meal_health(
  restaurant_name STRING COMMENT 'Name of the restaurant',
  meal_name STRING COMMENT 'Name or partial name of the meal item'
)
RETURNS TABLE(
  restaurant STRING COMMENT 'Restaurant name',
  item_name STRING COMMENT 'Menu item name',
  item_description STRING COMMENT 'Item description',
  food_category STRING COMMENT 'Food category',
  calories INT COMMENT 'Calories',
  total_fat DOUBLE COMMENT 'Total fat (g)',
  saturated_fat DOUBLE COMMENT 'Saturated fat (g)',
  trans_fat DOUBLE COMMENT 'Trans fat (g)',
  cholesterol INT COMMENT 'Cholesterol (mg)',
  sodium INT COMMENT 'Sodium (mg)',
  carbohydrates INT COMMENT 'Carbohydrates (g)',
  dietary_fiber INT COMMENT 'Dietary fiber (g)',
  sugar INT COMMENT 'Sugar (g)',
  protein INT COMMENT 'Protein (g)',
  health_score INT COMMENT 'Overall health score (0-100, higher is better)',
  health_assessment STRING COMMENT 'Health assessment text',
  concerns STRING COMMENT 'Health concerns identified'
)
COMMENT 'Analyzes the nutritional content of a specific meal and provides a health assessment with concerns'
RETURN
  WITH meal_data AS (
    SELECT 
      restaurant,
      item_name,
      item_description,
      food_category,
      CAST(calories AS INT) as calories,
      CAST(total_fat AS DOUBLE) as total_fat,
      CAST(saturated_fat AS DOUBLE) as saturated_fat,
      CAST(trans_fat AS DOUBLE) as trans_fat,
      CAST(cholesterol AS INT) as cholesterol,
      CAST(sodium AS INT) as sodium,
      CAST(carbohydrates AS INT) as carbohydrates,
      CAST(dietary_fiber AS INT) as dietary_fiber,
      CAST(sugar AS INT) as sugar,
      CAST(protein AS INT) as protein
    FROM main.default.restaurant_nutrition_data
    WHERE LOWER(REGEXP_REPLACE(restaurant, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(restaurant_name, '[^a-zA-Z0-9]', '')), '%')
      AND LOWER(REGEXP_REPLACE(item_name, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(meal_name, '[^a-zA-Z0-9]', '')), '%')
      AND calories != 'nan'
  )
  SELECT 
    restaurant,
    item_name,
    item_description,
    food_category,
    calories,
    total_fat,
    saturated_fat,
    trans_fat,
    cholesterol,
    sodium,
    carbohydrates,
    dietary_fiber,
    sugar,
    protein,
    -- Calculate health score (0-100)
    CAST(
      100 - 
      LEAST(30, GREATEST(0, (calories - 500) / 20)) -
      LEAST(20, GREATEST(0, (sodium - 1000) / 100)) -
      LEAST(20, saturated_fat * 2) -
      LEAST(15, trans_fat * 10) -
      LEAST(15, GREATEST(0, (sugar - 10) / 2))
      AS INT
    ) as health_score,
    -- Health assessment based on score
    CASE 
      WHEN 100 - LEAST(30, GREATEST(0, (calories - 500) / 20)) - LEAST(20, GREATEST(0, (sodium - 1000) / 100)) - LEAST(20, saturated_fat * 2) - LEAST(15, trans_fat * 10) - LEAST(15, GREATEST(0, (sugar - 10) / 2)) >= 80 THEN 'Excellent - Very healthy choice'
      WHEN 100 - LEAST(30, GREATEST(0, (calories - 500) / 20)) - LEAST(20, GREATEST(0, (sodium - 1000) / 100)) - LEAST(20, saturated_fat * 2) - LEAST(15, trans_fat * 10) - LEAST(15, GREATEST(0, (sugar - 10) / 2)) >= 60 THEN 'Good - Reasonably healthy'
      WHEN 100 - LEAST(30, GREATEST(0, (calories - 500) / 20)) - LEAST(20, GREATEST(0, (sodium - 1000) / 100)) - LEAST(20, saturated_fat * 2) - LEAST(15, trans_fat * 10) - LEAST(15, GREATEST(0, (sugar - 10) / 2)) >= 40 THEN 'Fair - Moderate concerns'
      ELSE 'Poor - Multiple health concerns'
    END as health_assessment,
    -- Identify specific concerns
    CONCAT_WS('; ', 
      CASE WHEN calories > 800 THEN 'High calories' END,
      CASE WHEN sodium > 1500 THEN 'High sodium' END,
      CASE WHEN saturated_fat > 8 THEN 'High saturated fat' END,
      CASE WHEN trans_fat > 0.5 THEN 'Contains trans fat' END,
      CASE WHEN sugar > 20 THEN 'High sugar' END,
      CASE WHEN dietary_fiber < 3 THEN 'Low fiber' END
    ) as concerns
  FROM meal_data
  ORDER BY health_score DESC;

In [0]:
#Test the tool
display(spark.sql("""
SELECT *
FROM main.default.analyze_meal_health('McDonald''s', 'Big Mac')
LIMIT 5
"""))

restaurant,item_name,item_description,food_category,calories,total_fat,saturated_fat,trans_fat,cholesterol,sodium,carbohydrates,dietary_fiber,sugar,protein,health_score,health_assessment,concerns
McDonald's,Big Mac,"Big Mac, Sandwiches",Burgers,540,29.0,10.0,1.5,75,1040,45,3,9,25,62,Good - Reasonably healthy,High saturated fat; Contains trans fat


In [0]:
%sql
--SQL Tool 3
CREATE OR REPLACE FUNCTION main.default.compare_two_meals_complete(
  restaurant1 STRING COMMENT 'First restaurant name',
  meal1 STRING COMMENT 'First meal name',
  restaurant2 STRING COMMENT 'Second restaurant name',
  meal2 STRING COMMENT 'Second meal name'
)
RETURNS TABLE(
  comparison_aspect STRING COMMENT 'What is being compared',
  meal_1_value STRING COMMENT 'Value for first meal',
  meal_2_value STRING COMMENT 'Value for second meal',
  winner STRING COMMENT 'Which meal is better for this aspect',
  difference STRING COMMENT 'Difference between the two'
)
COMMENT 'Side-by-side comparison of two meals including nutrition and calories. Shows which meal is healthier across multiple dimensions.'
RETURN
  WITH meal1_data AS (
    SELECT 
      item_name,
      CAST(calories AS INT) as calories,
      CAST(sodium AS INT) as sodium,
      CAST(protein AS INT) as protein,
      CAST(sugar AS INT) as sugar,
      CAST(saturated_fat AS DOUBLE) as saturated_fat
    FROM main.default.restaurant_nutrition_data
    WHERE LOWER(REGEXP_REPLACE(restaurant, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(restaurant1, '[^a-zA-Z0-9]', '')), '%')
      AND LOWER(REGEXP_REPLACE(item_name, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(meal1, '[^a-zA-Z0-9]', '')), '%')
      AND calories != 'nan'
    LIMIT 1
  ),
  meal2_data AS (
    SELECT 
      item_name,
      CAST(calories AS INT) as calories,
      CAST(sodium AS INT) as sodium,
      CAST(protein AS INT) as protein,
      CAST(sugar AS INT) as sugar,
      CAST(saturated_fat AS DOUBLE) as saturated_fat
    FROM main.default.restaurant_nutrition_data
    WHERE LOWER(REGEXP_REPLACE(restaurant, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(restaurant2, '[^a-zA-Z0-9]', '')), '%')
      AND LOWER(REGEXP_REPLACE(item_name, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(meal2, '[^a-zA-Z0-9]', '')), '%')
      AND calories != 'nan'
    LIMIT 1
  )
  SELECT 'Meal Name' as comparison_aspect,
         (SELECT item_name FROM meal1_data) as meal_1_value,
         (SELECT item_name FROM meal2_data) as meal_2_value,
         'N/A' as winner,
         'Comparing these two meals' as difference
  UNION ALL
  SELECT 'Calories',
         CAST((SELECT calories FROM meal1_data) AS STRING),
         CAST((SELECT calories FROM meal2_data) AS STRING),
         CASE WHEN (SELECT calories FROM meal1_data) < (SELECT calories FROM meal2_data) 
              THEN 'Meal 1' ELSE 'Meal 2' END,
         CONCAT(CAST(ABS((SELECT calories FROM meal1_data) - (SELECT calories FROM meal2_data)) AS STRING), ' cal difference')
  UNION ALL
  SELECT 'Sodium (mg)',
         CAST((SELECT sodium FROM meal1_data) AS STRING),
         CAST((SELECT sodium FROM meal2_data) AS STRING),
         CASE WHEN (SELECT sodium FROM meal1_data) < (SELECT sodium FROM meal2_data) 
              THEN 'Meal 1' ELSE 'Meal 2' END,
         CONCAT(CAST(ABS((SELECT sodium FROM meal1_data) - (SELECT sodium FROM meal2_data)) AS STRING), ' mg difference')
  UNION ALL
  SELECT 'Protein (g)',
         CAST((SELECT protein FROM meal1_data) AS STRING),
         CAST((SELECT protein FROM meal2_data) AS STRING),
         CASE WHEN (SELECT protein FROM meal1_data) > (SELECT protein FROM meal2_data) 
              THEN 'Meal 1' ELSE 'Meal 2' END,
         CONCAT(CAST(ABS((SELECT protein FROM meal1_data) - (SELECT protein FROM meal2_data)) AS STRING), ' g difference')
  UNION ALL
  SELECT 'Sugar (g)',
         CAST((SELECT sugar FROM meal1_data) AS STRING),
         CAST((SELECT sugar FROM meal2_data) AS STRING),
         CASE WHEN (SELECT sugar FROM meal1_data) < (SELECT sugar FROM meal2_data) 
              THEN 'Meal 1' ELSE 'Meal 2' END,
         CONCAT(CAST(ABS((SELECT sugar FROM meal1_data) - (SELECT sugar FROM meal2_data)) AS STRING), ' g difference')
  UNION ALL
  SELECT 'Saturated Fat (g)',
         CAST((SELECT saturated_fat FROM meal1_data) AS STRING),
         CAST((SELECT saturated_fat FROM meal2_data) AS STRING),
         CASE WHEN (SELECT saturated_fat FROM meal1_data) < (SELECT saturated_fat FROM meal2_data) 
              THEN 'Meal 1' ELSE 'Meal 2' END,
         CONCAT(CAST(ABS((SELECT saturated_fat FROM meal1_data) - (SELECT saturated_fat FROM meal2_data)) AS STRING), ' g difference')
  UNION ALL
  SELECT 'Overall Recommendation',
         CASE 
           WHEN (SELECT calories FROM meal1_data) < (SELECT calories FROM meal2_data) 
                AND (SELECT sodium FROM meal1_data) < (SELECT sodium FROM meal2_data)
           THEN 'Better Choice'
           ELSE 'Good Option'
         END,
         CASE 
           WHEN (SELECT calories FROM meal2_data) < (SELECT calories FROM meal1_data) 
                AND (SELECT sodium FROM meal2_data) < (SELECT sodium FROM meal1_data)
           THEN 'Better Choice'
           ELSE 'Good Option'
         END,
         CASE 
           WHEN (SELECT calories FROM meal1_data) + (SELECT sodium FROM meal1_data) / 10
                < (SELECT calories FROM meal2_data) + (SELECT sodium FROM meal2_data) / 10
           THEN 'Meal 1 Overall'
           ELSE 'Meal 2 Overall'
         END,
         'Based on calories and sodium';

In [0]:
#Test the tool
display(spark.sql("""
SELECT *
FROM main.default.compare_two_meals_complete('McDonald''s', 'Big Mac', 'Burger King', 'Whopper')
"""))

comparison_aspect,meal_1_value,meal_2_value,winner,difference
Meal Name,Big Mac,"Build Your Own Sandwich, Whopper (Plain)",N/A,Comparing these two meals
Calories,540,730,Meal 1,190 cal difference
Sodium (mg),1040,660,Meal 2,380 mg difference
Protein (g),25,49,Meal 2,24 g difference
Sugar (g),9,8,Meal 2,1 g difference
Saturated Fat (g),10.0,17.0,Meal 1,7.0 g difference
Overall Recommendation,Good Option,Good Option,Meal 1 Overall,Based on calories and sodium


In [0]:
%sql
--SQL Tool 4
CREATE OR REPLACE FUNCTION main.default.find_nutritious_meals(
  restaurant_filter STRING COMMENT 'Optional restaurant name (use empty string for all)',
  category_filter STRING COMMENT 'Optional food category (e.g., Salads, Sandwiches) - use empty string for all',
  max_calories INT COMMENT 'Maximum calories allowed'
)
RETURNS TABLE(
  restaurant STRING COMMENT 'Restaurant name',
  item_name STRING COMMENT 'Menu item name',
  food_category STRING COMMENT 'Food category',
  calories INT COMMENT 'Calories',
  sodium INT COMMENT 'Sodium (mg)',
  protein INT COMMENT 'Protein (g)',
  dietary_fiber INT COMMENT 'Dietary fiber (g)',
  sugar INT COMMENT 'Sugar (g)',
  nutrition_score INT COMMENT 'Nutrition score (0-100, higher is better)',
  rating STRING COMMENT 'Nutrition rating'
)
COMMENT 'Finds meals with good nutrition profiles based on calories, sodium, sugar, protein, and fiber content.'
RETURN
  WITH meal_base AS (
    SELECT 
      restaurant,
      item_name,
      item_description,
      food_category,
      TRY_CAST(calories AS INT) as calories,
      TRY_CAST(sodium AS INT) as sodium,
      TRY_CAST(protein AS INT) as protein,
      TRY_CAST(dietary_fiber AS INT) as dietary_fiber,
      TRY_CAST(sugar AS INT) as sugar
    FROM main.default.restaurant_nutrition_data
    WHERE TRY_CAST(calories AS INT) IS NOT NULL
      AND TRY_CAST(sodium AS INT) IS NOT NULL
      AND TRY_CAST(protein AS INT) IS NOT NULL
      AND TRY_CAST(dietary_fiber AS INT) IS NOT NULL
      AND TRY_CAST(sugar AS INT) IS NOT NULL
      AND TRY_CAST(calories AS INT) <= max_calories
      AND (restaurant_filter = '' OR LOWER(REGEXP_REPLACE(restaurant, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(restaurant_filter, '[^a-zA-Z0-9]', '')), '%'))
      AND (category_filter = '' OR LOWER(REGEXP_REPLACE(food_category, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(category_filter, '[^a-zA-Z0-9]', '')), '%'))
  )
  SELECT 
    restaurant,
    item_name,
    food_category,
    calories,
    sodium,
    protein,
    dietary_fiber,
    sugar,
    CAST(
      100 - 
      LEAST(25, GREATEST(0, (calories - 400) / 20)) -
      LEAST(25, GREATEST(0, (sodium - 800) / 100)) -
      LEAST(20, GREATEST(0, (sugar - 8) / 2)) +
      LEAST(10, protein / 5) +
      LEAST(10, dietary_fiber * 2)
      AS INT
    ) as nutrition_score,
    CASE 
      WHEN sodium < 600 AND sugar < 10 AND protein >= 20 AND dietary_fiber >= 5
        THEN 'Excellent - Highly Nutritious'
      WHEN sodium < 800 AND sugar < 15 AND protein >= 15
        THEN 'Very Good - Nutritious Choice'
      WHEN sodium < 1200 AND sugar < 20
        THEN 'Good - Moderate Nutrition'
      WHEN sodium < 1500
        THEN 'Fair - Higher Sodium/Sugar'
      ELSE 'Not Recommended - High Sodium/Sugar'
    END as rating
  FROM meal_base
  ORDER BY 
    CAST(
      100 - 
      LEAST(25, GREATEST(0, (calories - 400) / 20)) -
      LEAST(25, GREATEST(0, (sodium - 800) / 100)) -
      LEAST(20, GREATEST(0, (sugar - 8) / 2)) +
      LEAST(10, protein / 5) +
      LEAST(10, dietary_fiber * 2)
      AS INT
    ) DESC
  LIMIT 25;

In [0]:
#Test the tool
display(spark.sql("""
SELECT *
FROM main.default.find_nutritious_meals('', '', 600)
LIMIT 10
"""))

restaurant,item_name,food_category,calories,sodium,protein,dietary_fiber,sugar,nutrition_score,rating
El Pollo Loco,Double Chicken Avocado Salad,Salads,350,840,51,5,6,119,Good - Moderate Nutrition
Moe's Southwest Grill,"Fish, Double",Toppings & Ingredients,212,212,80,4,4,118,Very Good - Nutritious Choice
Whataburger,Garden Salad with Whatachick’n,Salads,400,700,35,5,4,117,Very Good - Nutritious Choice
Olive Garden,Herb-Grilled Salmon Coho (Regional),Entrees,400,840,50,4,3,117,Good - Moderate Nutrition
Olive Garden,Herb-Grilled Salmon Coho (Regional),Entrees,400,840,50,4,3,117,Good - Moderate Nutrition
Red Robin,Simply Grilled Chicken Salad,Salads,280,870,35,6,7,116,Good - Moderate Nutrition
Whataburger,Garden Salad with Grilled Chicken,Salads,290,770,34,6,4,116,Very Good - Nutritious Choice
Captain D's,Grilled Tilapia Salad,Salads,320,700,41,4,5,116,Very Good - Nutritious Choice
Carrabba's Italian Grill,Kids Grilled Shrimp and Steamed Broccoli,Entrees,160,380,30,7,3,116,Excellent - Highly Nutritious
Moe's Southwest Grill,"Steak, Double",Toppings & Ingredients,208,996,60,4,0,116,Good - Moderate Nutrition


In [0]:
%sql
--SQL Tool 5
CREATE OR REPLACE FUNCTION main.default.find_healthier_meal_alternatives(
  restaurant_name STRING COMMENT 'Restaurant to search (use empty string for all restaurants)',
  food_category STRING COMMENT 'Food category to search (e.g., Sandwiches, Salads) - use empty string for all categories',
  max_calories STRING COMMENT 'Maximum calories allowed (accepts string or number)',
  max_sodium STRING COMMENT 'Maximum sodium (mg) allowed (accepts string or number)'
)
RETURNS TABLE(
  restaurant STRING COMMENT 'Restaurant name',
  item_name STRING COMMENT 'Menu item name',
  food_category STRING COMMENT 'Food category',
  calories INT COMMENT 'Calories',
  total_fat DOUBLE COMMENT 'Total fat (g)',
  saturated_fat DOUBLE COMMENT 'Saturated fat (g)',
  sodium INT COMMENT 'Sodium (mg)',
  sugar INT COMMENT 'Sugar (g)',
  protein INT COMMENT 'Protein (g)',
  dietary_fiber INT COMMENT 'Dietary fiber (g)',
  health_score INT COMMENT 'Overall health score (higher is better)'
)
COMMENT 'Finds healthier meal options based on calorie and sodium limits. Returns meals ordered by health score.'
RETURN
  WITH meal_scores AS (
    SELECT 
      restaurant,
      item_name,
      food_category,
      TRY_CAST(calories AS INT) as calories,
      TRY_CAST(total_fat AS DOUBLE) as total_fat,
      TRY_CAST(saturated_fat AS DOUBLE) as saturated_fat,
      TRY_CAST(sodium AS INT) as sodium,
      TRY_CAST(sugar AS INT) as sugar,
      TRY_CAST(protein AS INT) as protein,
      TRY_CAST(dietary_fiber AS INT) as dietary_fiber,
      -- Calculate health score
      CAST(
        100 - 
        LEAST(30, GREATEST(0, (TRY_CAST(calories AS INT) - 500) / 20)) -
        LEAST(20, GREATEST(0, (TRY_CAST(sodium AS INT) - 1000) / 100)) -
        LEAST(20, TRY_CAST(saturated_fat AS DOUBLE) * 2) -
        LEAST(15, TRY_CAST(trans_fat AS DOUBLE) * 10) -
        LEAST(15, GREATEST(0, (TRY_CAST(sugar AS INT) - 10) / 2))
        AS INT
      ) as health_score
    FROM main.default.restaurant_nutrition_data
    WHERE TRY_CAST(calories AS INT) IS NOT NULL
      AND TRY_CAST(total_fat AS DOUBLE) IS NOT NULL
      AND TRY_CAST(saturated_fat AS DOUBLE) IS NOT NULL
      AND TRY_CAST(sodium AS INT) IS NOT NULL
      AND TRY_CAST(sugar AS INT) IS NOT NULL
      AND TRY_CAST(protein AS INT) IS NOT NULL
      AND TRY_CAST(dietary_fiber AS INT) IS NOT NULL
      AND TRY_CAST(trans_fat AS DOUBLE) IS NOT NULL
      AND TRY_CAST(calories AS INT) <= TRY_CAST(max_calories AS INT)
      AND TRY_CAST(sodium AS INT) <= TRY_CAST(max_sodium AS INT)
      AND (restaurant_name = '' OR LOWER(REGEXP_REPLACE(restaurant, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(restaurant_name, '[^a-zA-Z0-9]', '')), '%'))
      AND (food_category = '' OR LOWER(REGEXP_REPLACE(main.default.restaurant_nutrition_data.food_category, '[^a-zA-Z0-9]', '')) LIKE CONCAT('%', LOWER(REGEXP_REPLACE(food_category, '[^a-zA-Z0-9]', '')), '%'))
  )
  SELECT 
    restaurant,
    item_name,
    food_category,
    calories,
    total_fat,
    saturated_fat,
    sodium,
    sugar,
    protein,
    dietary_fiber,
    health_score
  FROM meal_scores
  ORDER BY health_score DESC, calories ASC
  LIMIT 20;

In [0]:
#Test the tool
display(spark.sql("""
SELECT *
FROM main.default.find_healthier_meal_alternatives('', '', 600, 800)
LIMIT 10
"""))

restaurant,item_name,food_category,calories,total_fat,saturated_fat,sodium,sugar,protein,dietary_fiber,health_score
Red Lobster,Iced Tea,Beverages,0,0.0,0.0,15,0,0,0,100
Golden Corral,"Gold Peak Tea, Unsweetened, 12 oz",Beverages,0,0.0,0.0,40,0,0,0,100
Subway,Cucumbers (3 slices),Toppings & Ingredients,0,0.0,0.0,0,0,0,0,100
Golden Corral,"Diet Coke, 32 oz",Beverages,0,0.0,0.0,130,0,0,0,100
California Pizza Kitchen,San Pellegrino Sparkling,Beverages,0,0.0,0.0,20,0,0,0,100
Golden Corral,"Coca Cola Zero Sugar, 12 oz",Beverages,0,0.0,0.0,40,0,0,0,100
Subway,Hot Pepper Relish,Toppings & Ingredients,0,0.0,0.0,170,0,0,0,100
Famous Dave's,"Jalapeños, For Build Your Own Burger or Sandwich",Toppings & Ingredients,0,0.0,0.0,0,0,0,0,100
Culver's,Yellow Mustard Packet,Toppings & Ingredients,0,0.0,0.0,130,0,0,0,100
Golden Corral,Coffee,Beverages,0,0.0,0.0,0,0,0,0,100


In [0]:
%sql
--SQL Tool 6

CREATE OR REPLACE FUNCTION main.default.analyze_ingredients_batch(ingredient_list STRING)
RETURNS STRING
COMMENT 'Analyzes a comma-separated list of additives and returns JSON with health scores'
RETURN
  WITH ingredients AS (
    SELECT 
      TRIM(value) AS ingredient,
      ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS idx
    FROM (SELECT EXPLODE(SPLIT(ingredient_list, ',')) AS value)
  ),
  matched AS (
    SELECT 
      i.ingredient,
      f.Substance AS matched_substance,
      f.Health_Risk_Score_1_to_10 AS health_risk_score,
      f.Health_Risk_Level AS health_risk_level,
      f.Risk_Categories AS risk_categories,
      f.Suggested_Alternatives AS suggested_alternatives
    FROM ingredients i
    LEFT JOIN main.default.fda_food_additives f
      ON UPPER(f.Substance) LIKE CONCAT('%', UPPER(i.ingredient), '%')
         OR UPPER(i.ingredient) LIKE CONCAT('%', UPPER(f.Substance), '%')
    QUALIFY ROW_NUMBER() OVER (PARTITION BY i.ingredient ORDER BY 
      CASE 
        WHEN UPPER(f.Substance) = UPPER(i.ingredient) THEN 1
        WHEN UPPER(f.Substance) LIKE CONCAT(UPPER(i.ingredient), '%') THEN 2
        WHEN UPPER(f.Substance) LIKE CONCAT('%', UPPER(i.ingredient), '%') THEN 3
        ELSE 4
      END
    ) = 1
  ),
  aggregated AS (
    SELECT
      COLLECT_LIST(
        STRUCT(
          ingredient,
          matched_substance,
          health_risk_score,
          health_risk_level,
          risk_categories,
          suggested_alternatives
        )
      ) AS details,
      AVG(CASE WHEN health_risk_score IS NOT NULL THEN health_risk_score END) AS avg_score,
      COUNT(CASE WHEN health_risk_score IS NOT NULL THEN 1 END) AS matched_count
    FROM matched
  )
  SELECT TO_JSON(
    STRUCT(
      ROUND(avg_score, 1) AS overall_score,
      CASE 
        WHEN avg_score IS NULL THEN 'No FDA-listed additives matched'
        WHEN avg_score <= 3 THEN 'Very Safe'
        WHEN avg_score <= 5 THEN 'Safe'
        WHEN avg_score <= 7 THEN 'Moderate Concern'
        ELSE 'High Concern'
      END AS overall_risk_level,
      details
    )
  )
  FROM aggregated;

In [0]:
#Test the tool
display(spark.sql("""
SELECT main.default.analyze_ingredients_batch('dextrin, salt, aceton') AS result
"""))

result
"{""overall_score"":4.3,""overall_risk_level"":""Safe"",""details"":[{""ingredient"":""aceton"",""matched_substance"":""ACETONE"",""health_risk_score"":4,""health_risk_level"":""Safe"",""risk_categories"":""General_Approved"",""suggested_alternatives"":""Natural plant extracts, essential oils, or fermentation-derived flavors""},{""ingredient"":""dextrin"",""matched_substance"":""DEXTRIN"",""health_risk_score"":5,""health_risk_level"":""Moderate"",""risk_categories"":""General_Approved"",""suggested_alternatives"":""Stevia, Monk fruit extract, or Allulose""},{""ingredient"":""salt"",""matched_substance"":""SALTS OF FATTY ACIDS"",""health_risk_score"":4,""health_risk_level"":""Safe"",""risk_categories"":""Low_Concern"",""suggested_alternatives"":""Natural gums (guar, locust bean, xanthan), pectin, or agar agar""}]}"


---
## 4. Defining the Agent

###Define the LLMs Used

In [0]:
# Available foundation model endpoints
AVAILABLE_LLMS = [
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b"
]
# Set the LLM: by index (e.g. 0, 1, 2) or by name
CHOSEN_LLM_ENDPOINT = AVAILABLE_LLMS[0]  # change index or set CHOSEN_LLM_ENDPOINT = "your-endpoint-name"
print(f"Using LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

Using LLM endpoint: databricks-gpt-oss-120b


###Define the Agent

In [0]:
%%writefile better_food_agent.py
import json
from typing import Any, Callable, Generator, Optional
from uuid import uuid4
import warnings

import backoff
import mlflow
import openai
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, VectorSearchRetrieverTool
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from unitycatalog.ai.core.base import get_uc_function_client

############################################
# Define your LLM endpoint and system prompt (placeholder replaced by next cell)
############################################
LLM_ENDPOINT_NAME = "__LLM_ENDPOINT__"

SYSTEM_PROMPT = """You are Fast Food Health Analyzer.

Your job is to answer questions about:

* fast food nutrition
* ingredients
* food additives
* meal quality
* healthier menu choices

Use available tools to obtain facts. Always cite which tool you used and explain the results. Never invent data. Kindly reject irrelevant questions.

# CORE RULES

* Use the minimum number of tool calls needed.
* Prefer answering with partial verified data rather than continuing to search.
* Never invent:

  * nutrition values
  * ingredient lists
  * additive scores
  * FDA information
  * rankings
  * health claims

If data cannot be obtained, clearly say so.

# TOOL SELECTION

Nutrition comparison:

* main__default__compare_two_meals_complete

Meal health analysis:

* main__default__analyze_meal_health

Finding healthier fast-food options:

* main__default__find_nutritious_meals
* main__default__find_healthier_meal_alternatives

Ingredient lookup:

* get_ingredients

Additive analysis using FDA dataset:

* main__default__analyze_ingredients_batch

Single additive deep lookup in the FDA dataset:

* main__default__get_additive_info

Fallback web search:

* youcom_search

# ADDITIVE ANALYSIS

For additive-related questions:

1. Obtain ingredient lists using get_ingredients().
2. Extract only likely additives and processing ingredients.

Examples:

* preservatives
* artificial colors
* artificial sweeteners
* emulsifiers
* stabilizers
* anti-caking agents
* acidity regulators
* flavor enhancers
* artificial flavors

Ignore ordinary food ingredients such as:

* meat
* vegetables
* cheese
* flour
* water
* oil
* salt
* sugar
* spices

3. Analyze all identified additives using ONE call to:
   main__default__analyze_ingredients_batch()

4. Use main__default__get_additive_info() only when:

   * the user asks about a single additive
   * extra detail is needed on one specific additive

# FDA DATA RULES

Only report FDA data returned by tools.

If an additive returns:

* {}
* null
* no match

Report:

"Not found in FDA database"

Do not assign:

* health score
* risk score
* concern level
* safety rating

# TOOL BUDGET

Single additive question:

* maximum 2 tool calls

Single menu item:

* maximum 5 tool calls

Two-item comparison:

* maximum 8 tool calls

Ranking or recommendation request:

* maximum 10 tool calls

# RETRY RULES

A tool may be used multiple times for different items.

Allowed:

* get_ingredients for Item A
* get_ingredients for Item B
* get_ingredients for Item C

Not allowed:

* repeatedly querying the same item after failure

If a lookup fails:

1. Make at most one fallback attempt.
2. Then continue using available data.

# COMPARISON WORKFLOW

For questions comparing two menu items:

Example:
"Compare a Taco Bell Crunchwrap Supreme and a Wendy's Dave's Single."

Workflow:

1. compare_two_meals_complete()
2. get_ingredients() for item A
3. get_ingredients() for item B
4. extract additives
5. analyze_ingredients_batch() once
6. answer

Do not continue searching after step 5.

# RANKING WORKFLOW

For questions such as:

* healthiest
* cleanest
* best
* top options
* high-protein choices

Workflow:

1. find candidate meals
2. apply user constraints
3. keep at most 5 candidate items
4. obtain ingredient lists for those candidates
5. extract additives
6. analyze additives using ONE batch call
7. rank and answer

Never analyze more than 5 menu items.

# RESPONSE FORMAT

Comparisons:

Nutrition Comparison
Additive Profile
Overall Assessment

Rankings:

Top Choices
Why They Rank Well
Overall Assessment

Always mention uncertainty when data is missing.

End every answer with:

Tools used: [actual tools used]
"""


###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation
## To create and see usage examples of more tools, see
## https://docs.databricks.com/generative-ai/agent-framework/agent-tool.html
###############################################################################
class ToolInfo(BaseModel):
    """
    Class representing a tool for the agent.
    - "name" (str): The name of the tool.
    - "spec" (dict): JSON description of the tool (matches OpenAI Responses format)
    - "exec_fn" (Callable): Function that implements the tool logic
    """

    name: str
    spec: dict
    exec_fn: Callable


def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


TOOL_INFOS = []

# You can use UDFs in Unity Catalog as agent tools
UC_TOOL_NAMES = ["main.default.get_additive_info", 
                 "main.default.analyze_meal_health", 
                 "main.default.compare_two_meals_complete",
                 "main.default.find_nutritious_meals",
                 "main.default.find_healthier_meal_alternatives",
                 "main.default.analyze_ingredients_batch"]

# UC and MCP tools are added in the notebook (see "Add MCP tools and create agent" cell)
# to avoid slow initialization at module import time.

def _sanitize_tool_spec(spec: dict) -> dict:
    """Remove JSON schema keywords that some model endpoints reject, including invalid format values (non-string, empty string, non-serializable) in nested structures."""
    import copy
    import json
    spec = copy.deepcopy(spec)
    
    def sanitize_recursive(obj):
        """Recursively remove invalid format fields and unsupported keywords"""
        if isinstance(obj, dict):
            # Remove invalid format fields: non-string, empty string, or non-serializable
            if "format" in obj:
                fmt = obj["format"]
                if not isinstance(fmt, str) or fmt == "":
                    obj.pop("format")
                else:
                    try:
                        json.dumps(fmt)
                    except TypeError:
                        obj.pop("format")
            
            # Remove unsupported schema keywords based on type
            t = obj.get("type")
            if t == "string":
                for key in ("minLength", "maxLength", "pattern"):
                    obj.pop(key, None)
            elif t in ("integer", "number"):
                for key in ("minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum"):
                    obj.pop(key, None)
            elif t == "array":
                for key in ("minItems", "maxItems", "uniqueItems"):
                    obj.pop(key, None)
            
            # Recursively process all nested dicts and lists
            for k, v in list(obj.items()):
                if isinstance(v, (dict, list)):
                    obj[k] = sanitize_recursive(v)
        elif isinstance(obj, list):
            return [sanitize_recursive(item) if isinstance(item, (dict, list)) else item for item in obj]
        
        return obj
    
    sanitize_recursive(spec)
    return spec

class ToolCallingAgent(ResponsesAgent):
    """
    Class representing a tool-calling Agent
    """

    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        """Initializes the ToolCallingAgent with tools."""
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        """Returns tool specifications in the format OpenAI expects. Strips schema keywords (e.g. minLength) that some endpoints reject."""
        return [_sanitize_tool_spec(tool_info.spec) for tool_info in self._tools_dict.values()]

    def execute_tool(self, tool_name: str, args: dict) -> Any:
        """Executes the specified tool with the given arguments."""
        import difflib
        # UC rejects params like {'': '{}'} for no-arg functions; keep only valid keyed args
        sane_args = {k: v for k, v in (args or {}).items() if k and isinstance(k, str)}
        # Normalize: strip quotes and model tokens (e.g. <|channel|>commentary) appended to name
        name = tool_name.strip().strip('"').strip("'")
        if "<" in name:
            name = name.split("<")[0].strip()
        
        # Use explicit span context to ensure proper tracing
        with mlflow.start_span(name=f"Tool: {name}", span_type=SpanType.TOOL) as span:
            span.set_inputs({"tool_name": name, "args": sane_args})
            try:
                if name in self._tools_dict:
                    result = self._tools_dict[name].exec_fn(**sane_args)
                else:
                    # Fallback 1: if LLM called without prefix, match registered tools that end with this name
                    candidates = [k for k in self._tools_dict if k.endswith(name)]
                    if candidates:
                        result = self._tools_dict[max(candidates, key=len)].exec_fn(**sane_args)
                    else:
                        # Fallback 2: fuzzy match for typos (e.g. youcms_search -> youcom_search)
                        close_matches = difflib.get_close_matches(name, self._tools_dict.keys(), n=1, cutoff=0.8)
                        if close_matches:
                            matched_tool = close_matches[0]
                            print(f"[Fuzzy match] Auto-corrected '{name}' to '{matched_tool}'")
                            result = self._tools_dict[matched_tool].exec_fn(**sane_args)
                        else:
                            raise KeyError(f"Unknown tool: {tool_name!r}. Known tools: {list(self._tools_dict.keys())}")
                
                span.set_outputs(result)
                return result
            except Exception as e:
                span.set_status("ERROR")
                span.set_attributes({"error": str(e)})
                raise

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="PydanticSerializationUnexpectedValue")
            for chunk in self.model_serving_client.chat.completions.create(
                model=self.llm_endpoint,
                messages=to_chat_completions_input(messages),
                tools=self.get_tool_specs(),
                stream=True,
            ):
                chunk_dict = chunk.to_dict()              
                if len(chunk_dict.get("choices", [])) > 0:
                    yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        """
        Execute tool calls, add them to the running message history, and return a ResponsesStreamEvent w/ tool output
        """
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception as e:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 20,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            last_msg = messages[-1]
            if last_msg.get("role", None) == "assistant":
                return
            elif last_msg.get("type", None) == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages), aggregator=messages
                )

        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


# Log the model using MLflow (AGENT is created in the notebook after adding MCP tools)
mlflow.openai.autolog()

Overwriting better_food_agent.py


###Supply the LLM to the Agent

In [0]:
# Bake chosen LLM into better_food_agent.py (run "Choose LLM" first)
with open("better_food_agent.py") as f:
    content = f.read()
with open("better_food_agent.py", "w") as f:
    f.write(content.replace("__LLM_ENDPOINT__", CHOSEN_LLM_ENDPOINT))
print("better_food_agent.py updated with LLM_ENDPOINT_NAME =", CHOSEN_LLM_ENDPOINT)

better_food_agent.py updated with LLM_ENDPOINT_NAME = databricks-gpt-oss-120b


---
## 5. MCP Connection and Python Function

###Create the MCP Connection and MCP Python Function

In [0]:
#MCP Connection
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

MCP_CONNECTION_NAME = "youcom_connection"

workspace_client = WorkspaceClient()
mcp_client = DatabricksMCPClient(
    server_url=f"{workspace_client.config.host}/api/2.0/mcp/external/{MCP_CONNECTION_NAME}",
    workspace_client=workspace_client,
)

search_tool = None
for t in mcp_client.list_tools():
    name = t.name.lower()
    if "search" in name and "research" not in name:
        search_tool = t.name
print(f"Using You.com MCP tool: {search_tool}")


def youcom_search(query: str) -> str:
    """Web search via You.com. Returns short result snippets."""
    result = mcp_client.call_tool(search_tool, {"query": query})
    return result.content[0].text if result.content else "No result"


def has_content(results: str) -> bool:
    """True if the search returned substantial text (not an error or near-empty result)."""
    return len(results) > 300 and "no result" not in results.lower()


def get_ingredients(restaurant: str, item: str) -> str:
    """Find the ingredient list for a menu item, best source first:
    1. The restaurant's own official website.
    2. fastfoodnutrition.org.
    3. A general web search across the whole web.
    """
    #1 the restaurant's official website
    official = youcom_search(f"{restaurant} official website {item} list of ingredients + additives")
    if has_content(official):
        return f"[Source: {restaurant} official website]\n{official}"

    #2 fastfoodnutrition.org
    print(f"[Tier 2] Official site had no ingredients — trying fastfoodnutrition.org for {restaurant} {item}...")
    ffn = youcom_search(f"{restaurant} {item} ingredients site:fastfoodnutrition.org")
    if has_content(ffn):
        return f"[Source: fastfoodnutrition.org]\n{ffn}"

    #3 the wider web
    print(f"[Tier 3] Searching the general web for {restaurant} {item}...")
    return f"[Source: general web search]\n" + youcom_search(f"{restaurant} {item} list of ingredients + additives")

print(get_ingredients("McDonald's", "Big Mac")[:1200])
print()
print(get_ingredients("Wendy's", "Baconator")[:1200])

Using You.com MCP tool: you-search
[Source: McDonald's official website]
Search Results for "McDonald's official website Big Mac list of ingredients + additives":

WEB RESULTS:

Title: Big Mac®: The Classic McDonald's Beef Burger
URL: https://www.mcdonalds.com/us/en-us/product/big-mac.html
Description: Grab a Big Mac® for a unique McDonald's experience: made with two all beef patties, Big Mac® sauce & more. Get yours today at a McDonald's near you!
Snippets:
- All nutrition information is based on average values for ingredients and is rounded in accordance with current U.S. FDA NLEA regulations. Variation in serving sizes, preparation techniques, product testing and sources of supply, as well as regional and seasonal differences may affect the nutrition values for each product. In addition, product formulations change periodically.
- We understand that each of our customers has individual needs and considerations when choosing a place to eat or drink outside their home, especially thos

###Add MCP Tools to the Agent and Create the Agent Instance

In [0]:
# Import the agent code from the file we just created
import sys
import importlib.util
import os

# Load the module directly from file path
notebook_dir = "/Workspace/Users/rittenvictoria@gmail.com/Final Project"
module_path = os.path.join(notebook_dir, "better_food_agent.py")

# Clear the module from cache if it exists
if "better_food_agent" in sys.modules:
    del sys.modules["better_food_agent"]

# Use importlib to load the module directly
spec = importlib.util.spec_from_file_location("better_food_agent", module_path)
better_food_agent = importlib.util.module_from_spec(spec)
sys.modules["better_food_agent"] = better_food_agent
spec.loader.exec_module(better_food_agent)

# Now import the classes/functions from the loaded module
from better_food_agent import ToolCallingAgent, ToolInfo, UC_TOOL_NAMES, create_tool_info
from databricks_openai import UCFunctionToolkit
from unitycatalog.ai.core.base import get_uc_function_client

# Clear and rebuild TOOL_INFOS to avoid duplicates on re-runs
TOOL_INFOS = []

# Initialize UC toolkit and add UC functions to TOOL_INFOS
print("Initializing Unity Catalog tools...")
uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))
print(f"Loaded {len(TOOL_INFOS)} UC tools")

# Create a tool spec for the MCP get_ingredients function
get_ingredients_spec = {
    "type": "function",
    "function": {
        "name": "get_ingredients",
        "description": "Find the ingredient list for a menu item from the restaurant's official website, fastfoodnutrition.org, or general web search.",
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant": {
                    "type": "string",
                    "description": "The restaurant name (e.g., 'McDonald's', 'Wendy's')"
                },
                "item": {
                    "type": "string",
                    "description": "The menu item name (e.g., 'Big Mac', 'Baconator')"
                }
            },
            "required": ["restaurant", "item"]
        }
    }
}

# Create a tool spec for generic You.com web search
youcom_search_spec = {
    "type": "function",
    "function": {
        "name": "youcom_search",
        "description": "Search the web using You.com for current information, research, facts, or general queries. Returns relevant search result snippets.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to look up on the web"
                }
            },
            "required": ["query"]
        }
    }
}

# Add both MCP tools to TOOL_INFOS
TOOL_INFOS.append(ToolInfo(
    name="get_ingredients",
    spec=get_ingredients_spec,
    exec_fn=get_ingredients  # Reference the function from cell 32
))

TOOL_INFOS.append(ToolInfo(
    name="youcom_search",
    spec=youcom_search_spec,
    exec_fn=youcom_search  # Reference the function from cell 32
))

print(f"Total tools registered: {len(TOOL_INFOS)}")
print(f"Tool names: {[t.name for t in TOOL_INFOS]}")

# Create the agent with all tools (UC + MCP)
AGENT = ToolCallingAgent(
    llm_endpoint=CHOSEN_LLM_ENDPOINT,
    tools=TOOL_INFOS
)

print(f"\nAgent created with LLM endpoint: {CHOSEN_LLM_ENDPOINT}")
print(f"Agent has {len(AGENT.get_tool_specs())} tools available")
import mlflow
mlflow.models.set_model(AGENT)

Initializing Unity Catalog tools...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Loaded 6 UC tools
Total tools registered: 8
Tool names: ['main__default__get_additive_info', 'main__default__analyze_meal_health', 'main__default__compare_two_meals_complete', 'main__default__find_nutritious_meals', 'main__default__find_healthier_meal_alternatives', 'main__default__analyze_ingredients_batch', 'get_ingredients', 'youcom_search']

Agent created with LLM endpoint: databricks-gpt-oss-120b
Agent has 8 tools available


---
## 6. Test the agent

In [0]:
# AGENT was created in the "Add MCP tools and create agent" cell above
import better_food_agent

# Inject uc_function_client into the module so create_tool_info can find it
better_food_agent.uc_function_client = uc_function_client

def answer(question: str) -> str:
    """Ask the agent and return the answer text."""
    from mlflow.types.responses import ResponsesAgentRequest
    resp = AGENT.predict(ResponsesAgentRequest(
        input=[{"role": "user", "content": question}]
    ))
    # Find the message item in the output (skip reasoning items)
    for item in resp.output:
        if item.type == "message" and hasattr(item, "content"):
            return item.content[0]["text"]
    return "No response generated"

print(answer("tell me about the additives within mcdonalds big mak"))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ll get_ingredients."}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...nt list additives".'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pyda

**Additive Profile – McDonald’s Big Mac**

| Additive (identified from the ingredient list) | FDA‑listed role | Health‑risk score* | Risk level** | Comments from FDA data |
|---|---|---|---|---|
| Calcium propionate | Preservative (prevents mold) | 4 | Safe | Generally regarded as safe; alternatives include rosemary extract or vinegar. |
| Sodium propionate | Preservative | 4 | Safe | Same safety profile as calcium propionate. |
| Sodium stearyl‑lactylate | Emulsifier / dough conditioner | – | – | No specific risk score returned (not a highlighted concern). |
| DATEM (diacetyl‑tartrate‑ester‑monoglyceride) | Dough conditioner / emulsifier | – | – | No specific risk score returned. |
| Ascorbic acid (Vitamin C) | Antioxidant, dough improver | 1 | Very Safe | Low‑concern antioxidant. |
| Azodicarbonamide | Dough conditioner (flour bleaching) | 9 | High Concern | Listed with “Cancer Risk, Moderate Concern”. Often replaced by safer alternatives. |
| Mono‑ and diglycerides of fatty acids | 

Trace(trace_id=tr-af8339c046ebfdd3c2c3d13aa60de956)

###Create Traces for Human Observation

In [0]:
TEST_QUERIES = [
    "Analyze a Chick-fil-A Spicy Deluxe Sandwich. Identify all synthetic additives, rank them by concern level, and clearly mark any additives that are not found in the FDA database.",    
    "Compare a Taco Bell Crunchwrap Supreme and a Wendy's Dave's Single. Show calories, sodium, additive profiles, and which meal appears cleaner overall.",
    "Find the three cleanest lunch items at Subway that have at least 25 grams of protein and explain why they rank well.",
    "Analyze the additives in this ingredient list: chicken, water, paprika extract, xanthan gum, lettuce, sodium benzoate, tomato, mono- and diglycerides, garlic powder, potassium sorbate. Only evaluate synthetic additives and report any that are not found in the FDA database.",
    "Who won the 2024 NBA Finals, and what were the key moments of the series?"
]

def _response_to_text(resp):
    """Extract final answer text from ResponsesAgentResponse."""
    text_parts = []
    for item in resp.output:
        c = getattr(item, "content", None)
        if c is None:
            continue
        for part in (c if isinstance(c, list) else [c]):
            if isinstance(part, dict) and "text" in part:
                text_parts.append(part["text"])
            elif hasattr(part, "text"):
                text_parts.append(part.text)
    return "\n".join(text_parts) if text_parts else str(resp.output)[:500]

# Run each test query across all LLMs (uses same tools; creates one agent per LLM)
# Requires "Add MCP tools and create agent" to have been run so agent.TOOL_INFOS exists
LLMS_TO_COMPARE = AVAILABLE_LLMS  # or subset, e.g. AVAILABLE_LLMS[:2]

all_results = []
for llm_name in LLMS_TO_COMPARE:
    print(f"\n{'='*60}\n  LLM: {llm_name}\n{'='*60}")
    agent_for_llm = better_food_agent.ToolCallingAgent(llm_endpoint=llm_name, tools=TOOL_INFOS)
    for q in TEST_QUERIES:
        print(f"\n--- Query: {q[:60]}{'...' if len(q) > 60 else ''} ---")
        resp = agent_for_llm.predict({
            "input": [{"role": "user", "content": q}],
            "custom_inputs": {"session_id": f"compare-{llm_name}"}
        })
        output_text = _response_to_text(resp)
        print(output_text[:400] + ("..." if len(output_text) > 400 else ""))
        all_results.append({"model": llm_name, "query": q, "output": output_text})

print(f"\nDone. Ran {len(TEST_QUERIES)} queries x {len(LLMS_TO_COMPARE)} LLMs = {len(all_results)} total.")


  LLM: databricks-gpt-oss-120b

--- Query: Analyze a Chick-fil-A Spicy Deluxe Sandwich. Identify all sy... ---


/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's....\n\nLet's do that."}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...omma separated.\n\n'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594:

**Additive Profile – Chick‑fil‑A Spicy Deluxe Sandwich**

| Additive (synthetic / processing) | FDA database match? | Health‑risk score* | Concern level** | Notes / Typical use |
|-----------------------------------|---------------------|-------------------|----------------|----------------------|
| **Sodium aluminum phosphate** | Found (matched) | 4 | Moderate | Leavening agent that helps the bun...

--- Query: Compare a Taco Bell Crunchwrap Supreme and a Wendy's Dave's ... ---


/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ne.\n\nProceed.\n\n"}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...gredients for each."}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pyda

Max iterations reached. Stopping.

--- Query: Find the three cleanest lunch items at Subway that have at l... ---


...

/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...t\'s call tool.\n\n'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments

**Additive Profile**

| Synthetic Additive | FDA Database Match | Health Risk Score | Risk Level |
|---------------------|--------------------|-------------------|------------|
| **Sodium Benzoate** | ✔ | 6 | Moderate |
| **Mono‑ and Diglycerides** | ✔ | 4 | Safe |
| **Potassium Sorbate** | ✔ | 4 | Low Concern |
| **Xanthan Gum** | ✔ | 3 | Safe |

**Overall Assessment**

All synthetic additives pr...

--- Query: Who won the 2024 NBA Finals, and what were the key moments o... ---


/local_disk0/.ephemeral_nfs/envs/pythonEnv-caa24de0-9644-4b6f-8a2a-128de87d71ca/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...can’t provide that.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


I’m sorry, but I can’t provide that.

Done. Ran 5 queries x 2 LLMs = 10 total.


[Trace(trace_id=tr-56ecff8626d8b228e7ff2b1aceda7b6b), Trace(trace_id=tr-3f7bd55c66417bd3c131b94a950e7e96), Trace(trace_id=tr-f6f90093f09c3a10eead5d969d1ccf7f), Trace(trace_id=tr-b9dc06454a54b701487e7bc21617c302), Trace(trace_id=tr-f0eb9ef882be9d9c7976c0c4143d2967), Trace(trace_id=tr-0964676c5e0df2acbedae345044f6e8a), Trace(trace_id=tr-2922e3b8a9374b1c0cb67ff093ab6b1f), Trace(trace_id=tr-030d655f6c46ce48c9640c014a066c8a), Trace(trace_id=tr-8d31e96832f5537b4632fbb3d3c99887), Trace(trace_id=tr-7951d1cc195b9ac622f35fa38d2ca3ae)]

##7. Evaluation

###Create a Manual Eval Dataset

In [0]:
# ---- Manual evaluation questions ----
# Each row = one trace. The 'inputs' column must be a dict whose keys match predict_fn's parameter names.
# Use a single parameter name (e.g. 'question') so MLflow calls predict_fn(question="...") once per row — not as one multi-turn session.
manual_eval_data = [
    "Analyze a Chick-fil-A Spicy Deluxe Sandwich. Identify all synthetic additives, rank them by concern level, and clearly mark any additives that are not found in the FDA database.",    
    "Compare a Taco Bell Crunchwrap Supreme and a Wendy's Dave's Single. Show calories, sodium, additive profiles, and which meal appears cleaner overall.",
    "Find the three cleanest lunch items at Subway that have at least 25 grams of protein and explain why they rank well.",
    "Analyze the additives in this ingredient list: chicken, water, paprika extract, xanthan gum, lettuce, sodium benzoate, tomato, mono- and diglycerides, garlic powder, potassium sorbate. Only evaluate synthetic additives and report any that are not found in the FDA database.",
    "Who won the 2024 NBA Finals, and what were the key moments of the series?"
]

print(f"Manual eval dataset: {len(manual_eval_data)} questions")

Manual eval dataset: 5 questions


In [0]:
# Combine into the final evaluation dataset
# (If synthetic generation above didn't work, that's OK — the manual set is sufficient)
import pandas as pd

# Convert list of questions to DataFrame with 'inputs' column containing dicts
# The key 'question' must match the predict_fn parameter name in cell 41
eval_data_df = pd.DataFrame({
    "inputs": [{"question": q} for q in manual_eval_data]
})
print(f"Total evaluation dataset: {len(eval_data_df)} questions")

# Create a UC-backed evaluation dataset and merge records (Databricks pattern:
# https://docs.databricks.com/aws/en/notebooks/source/mlflow3/evaluate-improve-genai-app.html)
uc_schema = "main.default"
evaluation_dataset_table_name = "betterfoodagent_expert_eval"
full_table_name = f"{uc_schema}.{evaluation_dataset_table_name}"

# Drop the table if it exists to avoid stale metadata issues
try:
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
except:
    pass

# Create a fresh dataset
eval_dataset = mlflow.genai.datasets.create_dataset(name=full_table_name)
eval_dataset = eval_dataset.merge_records(eval_data_df)
# Re-running this cell will append records again; for a fresh dataset, use a new table name or drop the table.
print(f"Evaluation dataset '{full_table_name}' ready ({len(eval_data_df)} records). Pass eval_dataset to mlflow.genai.evaluate().")

Total evaluation dataset: 5 questions
Evaluation dataset 'main.default.betterfoodagent_expert_eval' ready (5 records). Pass eval_dataset to mlflow.genai.evaluate().


###Define the Judge

In [0]:
# Predict function: one parameter per row so each row = one trace (not one multi-turn conversation).
# Parameter name must match the key in each row's inputs dict (e.g. {"inputs": {"question": "..."}}).
def predict_fn(question: str) -> str:
    """Run the agent on a single question and return the output string."""
    from mlflow.types.responses import ResponsesAgentRequest
    resp = AGENT.predict(ResponsesAgentRequest(
        input=[{"role": "user", "content": question}]
    ))
    # Find the message item in the output (skip reasoning items)
    for item in resp.output:
        if item.type == "message" and hasattr(item, "content"):
            return item.content[0]["text"]
    return "No response generated"

###Create a Custom LLM Judge

In [0]:
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers.base import ScorerSamplingConfig

# Create a custom judge that evaluates tool usage appropriateness
my_custom_judge = make_judge(
    name="better_food_judge",
    instructions="""You are evaluating an AI agent's ability to use tools appropriately and provide relevant answers.

The agent has access to these tools:
- main__default__compare_two_meals_complete: Compares two meals nutritionally. Does not check additives.
- main__default__analyze_meal_health: Analyzes nutrition and health profile of one meal.
- main__default__find_nutritious_meals: Finds fastfood meals with better nutrition profiles.
- main__default__find_healthier_meal_alternatives: Finds healthier meal alternatives using calorie and sodium limits.
- get_ingredients: Finds ingredient lists from the web.
- main__default__analyze_ingredients_batch: Analyzes a comma-separated list of additives using the FDA dataset.
- main__default__get_additive_info: Looks up one additive in detail.

User question: {{inputs}}
Agent trace: {{trace}}
Agent response: {{outputs}}

Evaluate: Did the agent (1) avoid redundant tool calls, (2) support claims with tool data, (3) reject out-of-domain questions, (4) not invent FDA data, (5) satisfy user constraints?

You MUST respond with a score from 1-5:
1 = Completely wrong (hallucinated data, wrong tools, ignored constraints, answered out-of-domain)
2 = Major issues (redundant calls, unsupported claims, partially wrong, missing key steps)
3 = Acceptable (correct but could be better, minor inefficiencies)
4 = Good (appropriate tools, accurate, satisfies constraints)
5 = Excellent (optimal tool usage, well-supported, exceeds expectations)

You MUST respond with valid JSON in this EXACT format:
{
  "result": 4,
  "reasoning": "One brief sentence explaining your score (max 15 words)"
}

IMPORTANT:
- Output ONLY the JSON object, nothing else
- Set "result" to an integer from 1 to 5
- Keep "reasoning" under 15 words
- Do not add explanatory text before or after the JSON
- If you cannot evaluate the trace for any reason, return: {"result": 3, "reasoning": "Unable to evaluate trace"}
- Always return valid JSON even if the trace is malformed or unclear

Example:
{"result": 4, "reasoning": "Agent used appropriate tools without redundancy or false claims."}""",
    model="databricks:/databricks-gpt-oss-120b", 
    feedback_value_type=int,
)

# Configure sampling to evaluate all records
#First time registering: my_custom_judge = my_custom_judge.register()
my_custom_judge = my_custom_judge.update(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

print(f"Custom judge '{my_custom_judge.name}' defined with model: {my_custom_judge.model}")
print(f"Feedback type: {my_custom_judge.feedback_value_type}")
print(f"Sample rate: 100%")


Custom judge 'better_food_judge' defined with model: databricks:/databricks-gpt-oss-120b
Feedback type: <class 'int'>
Sample rate: 100%


###Evaluate Using the Custom Judge

In [0]:
import mlflow
import better_food_agent
from unitycatalog.ai.core.base import get_uc_function_client

# Ensure uc_function_client is available in the better_food_agent module
# (required for AGENT's tools to work during evaluation)
if not hasattr(better_food_agent, 'uc_function_client'):
    better_food_agent.uc_function_client = get_uc_function_client()

print("Running evaluation with all judges...")
all_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        my_custom_judge,  # Your custom judge from the cell above; if you changed the judge name, update this
    ]
)

print("\nEvaluation complete. See the MLflow UI for the full results table.")

Running evaluation with all judges...


2026/06/20 18:01:47 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/5 [Elapsed: 00:00, Remaining: ?]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments are.
  check_function_info(function_info)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality fo


Evaluation complete. See the MLflow UI for the full results table.


###Create Trace Questions for Human Feedback

In [0]:
TRACE_QUESTIONS = [
    # get_additive_info
    "Tell me about Red 40. Is it safe?",
    # analyze_meal_health
    "Analyze the healthiness of a Whataburger Spicy Chicken meal with medium fries and a medium Coke.",
    # compare_two_meals_complete
    "Show a side-by-side nutrition comparison of a Chick-fil-A Original Chicken Sandwich and a Popeyes Classic Chicken Sandwich.",
    # analyze_ingredients_batch
    "Analyze these additives: sodium nitrite, BHT, caramel color, potassium sorbate, calcium propionate.",
    # find_nutritious_meals
    "Show me the most nutritious options at Chipotle.",
    # find_healthier_meal_alternatives
    "Find healthier burger options at Burger King that contain fewer than 600 calories and less than 900 mg of sodium.",
    # get_ingredients
    "What ingredients are in a Chick-fil-A Spicy Deluxe Sandwich?",
    # Multi-step or edge case
    "Find the three cleanest lunch items at Subway that contain at least 25 grams of protein and explain why they rank well.",
    # Complex question
    "Find me the cleanest breakfast item at McDonald's with at least 20g protein and no controversial additives."
]

# Run all queries and store results with the official MLflow tagging
trace_results = []
import mlflow
from mlflow.types.responses import ResponsesAgentRequest

# Ensure uc_function_client is available in the better_food_agent module
# (in case the module was reloaded after cell 31 ran)
import better_food_agent
from unitycatalog.ai.core.base import get_uc_function_client
if not hasattr(better_food_agent, 'uc_function_client'):
    better_food_agent.uc_function_client = get_uc_function_client()

for i, question in enumerate(TRACE_QUESTIONS):
    print(f"\n--- Trace {i+1}/9: {question[:60]}... ---")
    try:
        # Use the AGENT created earlier in the notebook
        resp = AGENT.predict(ResponsesAgentRequest(
            input=[{"role": "user", "content": question}]
        ))
        # Extract the message content from the response
        output = None
        for item in resp.output:
            if item.type == "message" and hasattr(item, "content"):
                output = item.content[0]["text"]
                break
        if output is None:
            output = "No response generated"
        
        print(f"Output: {output[:150]}...")
        trace_results.append({
            "question": question,
            "output": output,
            "error": None
        })
    except Exception as e:
        print(f"Error: {e}")
        trace_results.append({
            "question": question,
            "output": None,
            "error": str(e)
        })

print(f"\n=== Generated {len(trace_results)} traces with 120b model ===")

# Generate 6th trace with 20b model with an irrelevant question
print("\n--- Creating 10th trace with 20b model ---")
import better_food_agent
agent_20b = better_food_agent.ToolCallingAgent(
    llm_endpoint="databricks-gpt-oss-20b",
    tools=TOOL_INFOS
)

question_10 = "What's the most popular fastfood restaurant in Germany?"
print(f"\n--- Trace 10/10: {question_10[:60]}... ---")
try:
    resp = agent_20b.predict(ResponsesAgentRequest(
        input=[{"role": "user", "content": question_10}]
    ))
    output = None
    for item in resp.output:
        if item.type == "message" and hasattr(item, "content"):
            output = item.content[0]["text"]
            break
    if output is None:
        output = "No response generated"
    
    print(f"Output: {output[:150]}...")
    trace_results.append({
        "question": question_10,
        "output": output,
        "error": None
    })
except Exception as e:
    print(f"Error: {e}")
    trace_results.append({
        "question": question_10,
        "output": None,
        "error": str(e)
    })

print(f"\n=== Generated {len(trace_results)} traces total (9 with 120b, 1 with 20b) ===")


--- Trace 1/9: Tell me about Red 40. Is it safe?... ---
Output: **Red 40 (Allura Red AC)**  

- **FDA database result:** No specific entry was returned for “Red 40” from the FDA additive information tool (the query...

--- Trace 2/9: Analyze the healthiness of a Whataburger Spicy Chicken meal ... ---


/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments are.
  check_function_info(function_info)


Output: **Nutrition Comparison**

| Item | Calories | Total Fat (g) | Saturated Fat (g) | Sodium (mg) | Carbohydrates (g) | Sugar (g) | Protein (g) |
|------|...

--- Trace 3/9: Show a side-by-side nutrition comparison of a Chick-fil-A Or... ---
Output: Max iterations reached. Stopping....

--- Trace 4/9: Analyze these additives: sodium nitrite, BHT, caramel color,... ---


/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments are.
  check_function_info(function_info)


Output: **Additive Analysis**

| Additive | FDA‑Reported Match | Health Risk Score* | Health Risk Level | Risk Category | Suggested Alternatives |
|----------...

--- Trace 5/9: Show me the most nutritious options at Chipotle.... ---
Output: **Most Nutritious Chipotle Options (based on the *nutrition_score* returned by the “find_nutritious_meals” tool)**  

| Rank | Item (Category) | Calor...

--- Trace 6/9: Find healthier burger options at Burger King that contain fe... ---
Error: Invalid parameters provided: {'max_calories': "Parameter max_calories should be of type STRING (corresponding python type <class 'str'>), but got <class 'int'>", 'max_sodium': "Parameter max_sodium should be of type STRING (corresponding python type <class 'str'>), but got <class 'int'>"}.

--- Trace 7/9: What ingredients are in a Chick-fil-A Spicy Deluxe Sandwich?... ---
Output: **Chick‑fil‑A Spicy Deluxe Sandwich – Ingredient List**

*(sourced from Chick‑fil‑A’s official website via the `get_ingredients` t

/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments are.
  check_function_info(function_info)


Output: **Top 3 “cleanest” Subway lunch items with ≥ 25 g protein**

| # | Menu item (6‑inch or Bowl) | Protein (g) | Calories | Sodium (mg) | Why it’s a clea...

--- Trace 9/9: Find me the cleanest breakfast item at McDonald's with at le... ---


/local_disk0/.ephemeral_nfs/envs/pythonEnv-1fdc4c89-092e-4d1e-a7ea-9c87e3552570/lib/python3.12/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: ingredient_list for the function main.default.analyze_ingredients_batch. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments are.
  check_function_info(function_info)


Output: **Breakfast Item that meets your criteria**

| Item (McDonald’s) | Approx. Protein* | Additives (identified) |
|-------------------|------------------...

=== Generated 9 traces with 120b model ===

--- Creating 10th trace with 20b model ---

--- Trace 10/10: What's the most popular fastfood restaurant in Germany?... ---
Output: I’m sorry, but I can’t answer that question....

=== Generated 10 traces total (9 with 120b, 1 with 20b) ===


[Trace(trace_id=tr-ec64c307106f8361732fef14d5f796bf), Trace(trace_id=tr-938f6f2c9640d2e0bfeb0b138856310f), Trace(trace_id=tr-16c4e61a61648a5e6ec709960279f08e), Trace(trace_id=tr-f19c1435879d5433a6c95fb9502b4d05), Trace(trace_id=tr-44edf3cecd60156335f78f93156ee292), Trace(trace_id=tr-91f5aa4dc642de6ddd3f633ef87b6cc6), Trace(trace_id=tr-356fc377a773daa1c537ee72bb0744f7), Trace(trace_id=tr-90c67e1898df2ee2994b43fdfdda6367), Trace(trace_id=tr-c3045ed8c8c65f5923edad6229e713e0), Trace(trace_id=tr-906e9858ad1aa476444dd1a9ce40b65e)]

In [0]:
# Retrieve trace IDs from MLflow - get the most recent 10
traces = mlflow.search_traces(
    max_results=10,  # Updated to include 11th trace
    order_by=["timestamp_ms DESC"],
)

print(f"Found {len(traces)} recent traces in experiment.")


Found 10 recent traces in experiment.


In [0]:
# Assuming 'traces' already contains all relevant rows/columns
# If your 'inputs' column isn't named 'inputs' or 'response', rename as needed
answer_sheet_df = traces.rename(columns={'response': 'output'})  # MLflow usually expects 'output'
# If 'inputs' is nested, flatten as needed

result = mlflow.genai.evaluate(
    data=answer_sheet_df,
    scorers=[my_custom_judge] 
)

Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

2026/06/20 18:06:12 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'better_food_judge': 1/10 failed. Check individual trace assessments for detailed error messages.


In [0]:
# Tag each trace for alignment - REQUIRED for later filtering
import mlflow

trace_ids = traces['trace_id'].values

for trace_id in trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="eval", value="align")

print(f"Tagged {len(traces)} traces with eval: align tag.")

Tagged 10 traces with eval: align tag.


-------------
####This is where human gives a feedback in Experiments section.
-------------

###Align Human Feedback with the Judge

In [0]:
import mlflow

# Retrieve traces from the specific evaluation run "dapper-toad-624"
# First, find the run by name
runs = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'dapper-toad-624'",
    max_results=1
)

if len(runs) == 0:
    raise ValueError("Could not find evaluation run 'dapper-toad-624'. Check the run name.")

run_id = runs.iloc[0]['run_id']
experiment_id = runs.iloc[0]['experiment_id']
print(f"Found evaluation run: {run_id}")
print(f"Experiment ID: {experiment_id}")

# Retrieve traces associated with this evaluation run
# Option 1: Get traces tagged with this run
traces_for_alignment = mlflow.search_traces(
    experiment_ids=[experiment_id],
    filter_string=f"tag.eval = 'align'",  # Still filter by eval tag if you tagged them
    max_results=10,
    return_type="list",
    order_by=["timestamp_ms DESC"]  # Get the most recent ones
)

print(f"Found {len(traces_for_alignment)} traces with tag eval='align' from run 'dapper-toad-624'.")

Found evaluation run: a37bae38de814d4ca18bc9d0d1cf40a1
Experiment ID: 625648936364199


/home/spark-0a7ae129-f235-41b9-96a4-38/.ipykernel/10822/command-8935344952357127-713003277:20: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces_for_alignment = mlflow.search_traces(


Found 10 traces with tag eval='align' from run 'dapper-toad-624'.


###Retrieve Human and LLM Feedback

In [0]:
# Print the LLM judge assessment and the matching human feedback for quick grading.
def _to_dict_if_possible(obj):
    if isinstance(obj, dict):
        return obj
    for method_name in ("to_dict", "model_dump", "dict"):
        if hasattr(obj, method_name):
            try:
                return getattr(obj, method_name)()
            except Exception:
                pass
    return None


def _get_value(obj, *names):
    if obj is None:
        return None
    if isinstance(obj, dict):
        for name in names:
            if name in obj:
                return obj[name]
        return None
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    obj_dict = _to_dict_if_possible(obj)
    if isinstance(obj_dict, dict):
        for name in names:
            if name in obj_dict:
                return obj_dict[name]
    return None


def _normalize_text(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return str(value)
    value_dict = _to_dict_if_possible(value)
    if isinstance(value_dict, dict):
        for key in ("value", "boolean_value", "string_value", "text"):
            if key in value_dict:
                return str(value_dict[key])
    return str(value)


def _extract_assessments(trace):
    trace_dict = _to_dict_if_possible(trace) or {}
    candidates = [
        _get_value(trace, "assessments"),
        _get_value(_get_value(trace, "info"), "assessments"),
        _get_value(_get_value(trace, "data"), "assessments"),
        trace_dict.get("assessments"),
    ]
    if isinstance(trace_dict.get("info"), dict):
        candidates.append(trace_dict["info"].get("assessments"))
    if isinstance(trace_dict.get("data"), dict):
        candidates.append(trace_dict["data"].get("assessments"))

    for candidate in candidates:
        if candidate:
            return list(candidate)
    return []


def _extract_question(trace):
    trace_dict = _to_dict_if_possible(trace) or {}
    candidates = [
        _get_value(trace, "inputs"),
        _get_value(_get_value(trace, "data"), "inputs"),
        _get_value(_get_value(_get_value(trace, "data"), "request"), "inputs"),
        trace_dict.get("inputs"),
    ]
    if isinstance(trace_dict.get("data"), dict):
        candidates.append(trace_dict["data"].get("inputs"))
        if isinstance(trace_dict["data"].get("request"), dict):
            candidates.append(trace_dict["data"]["request"].get("inputs"))

    for candidate in candidates:
        if isinstance(candidate, dict):
            if "input" in candidate:
                return str(candidate["input"])
            if candidate:
                first_value = next(iter(candidate.values()))
                return str(first_value)
        if isinstance(candidate, str):
            return candidate
    return None


def _extract_trace_id(trace):
    return (
        _get_value(trace, "trace_id")
        or _get_value(_get_value(trace, "info"), "trace_id")
        or (_to_dict_if_possible(trace) or {}).get("trace_id")
        or "unknown-trace"
    )


def _assessment_bucket(assessment):
    source = _get_value(assessment, "source_type", "source", "source_id")
    source_name = _normalize_text(source).lower() if source is not None else ""
    if any(token in source_name for token in ("human", "manual", "feedback", "annotation", "label", "user")):
        return "human"
    return "llm"


def _assessment_summary(assessment):
    value = _normalize_text(_get_value(assessment, "value", "feedback", "result")) or "(missing)"
    rationale = _normalize_text(_get_value(assessment, "rationale", "justification", "explanation")) or "(missing)"
    return value, rationale


print("\n=== Pre-alignment assessment check ===")
JUDGE_NAME = 'better_food_judge'  # Match the judge name from cell 37
llm_present_count = 0
human_present_count = 0
both_present_count = 0
for idx, trace in enumerate(traces_for_alignment, start=1):
    relevant_assessments = [
        assessment
        for assessment in _extract_assessments(trace)
        if (_get_value(assessment, "name", "assessment_name", "key") or JUDGE_NAME) == JUDGE_NAME
    ]

    llm_assessment = next((a for a in relevant_assessments if _assessment_bucket(a) == "llm"), None)
    human_assessment = next((a for a in relevant_assessments if _assessment_bucket(a) == "human"), None)

    question = _extract_question(trace)
    question_preview = (question[:100] + "...") if question and len(question) > 100 else question

    print(f"\nTrace {idx}: {_extract_trace_id(trace)}")
    if question_preview:
        print(f"  Question: {question_preview}")

    llm_value, llm_rationale = _assessment_summary(llm_assessment)
    human_value, human_rationale = _assessment_summary(human_assessment)

    if llm_assessment is not None:
        llm_present_count += 1
    if human_assessment is not None:
        human_present_count += 1
    if llm_assessment is not None and human_assessment is not None:
        both_present_count += 1

    print(f"  LLM assessment: {llm_value}")
    print(f"  LLM rationale: {llm_rationale}")
    print(f"  Human assessment: {human_value}")
    print(f"  Human rationale: {human_rationale}")

print("\n=== Assessment completeness summary ===")
print(f"Traces with LLM assessments: {llm_present_count}/{len(traces_for_alignment)}")
print(f"Traces with human assessments: {human_present_count}/{len(traces_for_alignment)}")
print(f"Traces with both present: {both_present_count}/{len(traces_for_alignment)}")



=== Pre-alignment assessment check ===

Trace 1: tr-906e9858ad1aa476444dd1a9ce40b65e
  LLM assessment: 5.0
  LLM rationale: (missing)
  Human assessment: 5.0
  Human rationale: The agent correctly rejected the irrelevant question.

Trace 2: tr-c3045ed8c8c65f5923edad6229e713e0
  LLM assessment: 4.0
  LLM rationale: (missing)
  Human assessment: 3.0
  Human rationale: Acceptable, but the agent invoked too many tools. find_nutritious_meals tool can be utilized.

Trace 3: tr-90c67e1898df2ee2994b43fdfdda6367
  LLM assessment: 2.0
  LLM rationale: (missing)
  Human assessment: 3.0
  Human rationale: Acceptable, but the agent invoked too many tools. find_nutritious_meals tool can be utilized.

Trace 4: tr-356fc377a773daa1c537ee72bb0744f7
  LLM assessment: 5.0
  LLM rationale: (missing)
  Human assessment: 5.0
  Human rationale: The agent invoked the correct tool appropriate number of times.

Trace 5: tr-91f5aa4dc642de6ddd3f633ef87b6cc6
  LLM assessment: (missing)
  LLM rationale: (missing)
 

##8. Optimization of the Judge Using SIMBA

In [0]:
import logging
import sys

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
    stream=sys.stdout,
)

# Keep only SIMBA-related logs. Switch any of these to DEBUG if you need deeper troubleshooting.
logging.getLogger("SIMBAAlignmentOptimizer").setLevel(logging.INFO)
logging.getLogger("mlflow.genai.judges.optimizers.simba").setLevel(logging.INFO)
logging.getLogger("dspy.teleprompt.simba").setLevel(logging.INFO)

# Silence noisy internals
for name in [
    "mlflow",
    "mlflow.genai.judges.optimizers.dspy_utils",
    "databricks.sdk",
    "httpcore",
    "httpx",
    "urllib3",
]:
    logging.getLogger(name).setLevel(logging.WARNING)


In [0]:
# Required path: align the judge with SIMBA.
# Expect this to take several minutes. If you hit rate limits, rerun the cell or reduce max_steps.
# Alignment may improve the judge, but it is not guaranteed to outperform your original prompt on every trace.

# Align the judge using traces with SME feedback
from mlflow.genai.judges import make_judge
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer
optimizer = SIMBAAlignmentOptimizer(
    model="databricks:/databricks-gpt-oss-120b",  # Optimizer model used to propose judge revisions.
    simba_kwargs={
        "max_steps": 2,  # Each step runs another optimization round. Lower is faster.
        "max_demos": 2,  # Rule-only optimization. Set >0 to let SIMBA add few-shot demos.
    },
)
aligned_judge = my_custom_judge.align(traces=traces_for_alignment, optimizer=optimizer)
print(f"Alignment complete for judge: {aligned_judge.name}")

2026-06-20 21:52:36,047 INFO SIMBAAlignmentOptimizer: Preparing optimization with 10 examples from 10 traces


2026/06/20 21:52:36 INFO mlflow.genai.judges.optimizers.simba: Starting SIMBA optimization with 10 examples (set logging to DEBUG for detailed output)


Processed 22 / 60 examples:  37%|███▋      | 22/60 [01:16<03:35,  5.67s/it]

2026/06/20 21:53:53 WARNING dspy.teleprompt.simba_utils: Failed to parse response from judge model. Response: 


Processed 42 / 60 examples:  70%|███████   | 42/60 [02:06<01:08,  3.78s/it]

...

Processed 60 / 70 examples:  86%|████████▌ | 60/70 [05:40<00:31,  3.17s/it]

2026/06/20 22:09:28 WARNING dspy.teleprompt.simba_utils: Failed to parse response from judge model. Response: 


Processed 30 / 30 examples: 100%|██████████| 30/30 [02:24<00:00,  4.83s/it]

2026/06/20 22:12:58 INFO mlflow.genai.judges.optimizers.simba: SIMBA optimization completed



Alignment complete for judge: better_food_judge


###Compare the Original and Aligned Judge

In [0]:
print('ORIGINAL JUDGE:\n')
print(my_custom_judge.instructions)
print('\n')
print("_"*100)
print("\nALIGNED JUDGE\n")
print(aligned_judge.instructions)

ORIGINAL JUDGE:

You are evaluating an AI agent's ability to use tools appropriately and provide relevant answers.

The agent has access to these tools:
- main__default__compare_two_meals_complete: Compares two meals nutritionally. Does not check additives.
- main__default__analyze_meal_health: Analyzes nutrition and health profile of one meal.
- main__default__find_nutritious_meals: Finds fastfood meals with better nutrition profiles.
- main__default__find_healthier_meal_alternatives: Finds healthier meal alternatives using calorie and sodium limits.
- get_ingredients: Finds ingredient lists from the web.
- main__default__analyze_ingredients_batch: Analyzes a comma-separated list of additives using the FDA dataset.
- main__default__get_additive_info: Looks up one additive in detail.

User question: {{inputs}}
Agent trace: {{trace}}
Agent response: {{outputs}}

Evaluate: Did the agent (1) avoid redundant tool calls, (2) support claims with tool data, (3) reject out-of-domain questions,

In [0]:
print("\nRunning aligned judge for comparison...")

# Get experiment_id from traces_for_alignment (loaded in cell 48)
experiment_id = traces_for_alignment[0].info.experiment_id
print(f"Using experiment_id: {experiment_id}")

traces_for_eval = mlflow.search_traces(
    experiment_ids=[experiment_id],  # Specify the experiment to search in
    filter_string="tag.eval = 'align'",
    max_results=10,
)
answer_sheet_df_align = traces_for_eval.rename(columns={'response': 'output'})  # MLflow usually expects 'output'

aligned_results = mlflow.genai.evaluate(
    data=answer_sheet_df_align,
    scorers=[aligned_judge]
)



Running aligned judge for comparison...
Using experiment_id: 625648936364199


/home/spark-937d24c7-4d38-433f-b679-d0/.ipykernel/10072/command-8935344952357132-1550481098:7: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces_for_eval = mlflow.search_traces(


Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

2026/06/20 20:38:38 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'better_food_judge': 1/10 failed. Check individual trace assessments for detailed error messages.


#Project Architecture and Pipeline

**Architecture:**

Two Datasets: FDA food additives (health risk scores) + restaurant nutrition data (MenuStat 2022).

Six SQL Tools: Get additive info, analyze meals, compare meals, find nutritious options, find healthier alternatives, batch analyze ingredients.

One MCP Tool: You.com web search for ingredient lists using Python. Not registered to Unity Catalog due to it needing live MCP connection.

Agent: better_food_agent using databricks-gpt-oss-120b (with 20b for comparison).

**Pipeline:**

Setup (Cells 1-7): Install dependencies, download datasets, create Unity Catalog tables.

Tools (Cells 8-20): Create and test 6 SQL functions that query nutrition/additive data.

Agent (Cells 21-32): Define agent code in better_food_agent.py, configure LLM, add MCP connection for web search.

Testing (Cells 33-36): Run sample queries, create traces for observation.

Evaluation (Cells 37-56): Build eval dataset, create custom judge (better_food_judge), run evaluation, collect human feedback.

Optimization (Cells 57-62): Use SIMBA to align judge with human feedback, compare original vs. aligned judge performance.

#Agent Performance Analysis

###Manual Trace Observation

After the agent was created and tested with a simple query, 5 test queries were created to have an initial observation on the agent's performance, and these 5 traces were created with  databricks-gpt-oss-20b and databricks-gpt-oss-120b LLM models, adding up to a total of 10 traces. Initial manual observation showed that the irrelevant questions were rejected by the agent kindly, as desired by both LLM models. Second question about the additives was answered well by both LLM & agent combinations. The agent invoked the correct tool and did not make any redundant calls and used the tool only once and provided a thoughtful response. The cleanest lunch item at Subway was answered at an acceptable level by 20b model powered agent although it made some redundant calls, while 120b reached the maximum limit. Analysis of multiple additives was handled correctly by the 20b LLM powered agent, while 120b reached the limit again. Analyzing the Chick-fil-A Spicy Deluxe Sandwich and identifying its additives was answered well by both agents, but it did not exceed expectations. The agent is instructed to use the minimum necessary tools; hence, it usually avoids calling the same tool with a different verbiage. Some of the additives were actually in the FDA dataset, but were not found. This could be either fixed with multiple tool calls as long as the budget allows or using LLM reasoning to call different variations of the same additive.

This manual observation was not recorded in evaluations; the questions in the evaluations were chosen differently. Based on our unrecorded scoring, the average score for the 20b model was determined to be 4.2, while it is 3.2 for the 120b model, mainly because of reaching the maximum limit without providing an answer. Similarly, average token usage was 29576 for 20b, while it was 51689 for 120b. Comparing the two models from both response quality and cost perspectives, 20b provides a better return on investment.

###Judge Creation Process

Using the manual trace observation, the custom LLM judge was created with a special scoring system. The tools that the agent has access to were defined with a short description. The evaluation prompt checks if any redundant calls were made, if the agent supported claims with tool data, if it rejected irrelevant questions, if it invented any FDA data, and if it satisfied the user's query. The judge response guideline strictly defines the scoring system. Where 1 refers to the agent response being completely wrong, 2: major issues such as redundant calls or missing key steps, 3: acceptable but existing minor issues, 4: good, and 5: excellent. Due to some JSON format issues noticed during execution, a JSON format guideline was created. Some important instructions followed this guideline, including the required output type, ensuring that results are numeric, limiting reasoning to fewer than 15 words (to avoid execution errors), and specifying how the agent should return a result even when a failure occurs. The judge was created with databricks-gpt-oss-120b LLM model.

###Evaluation Process and Analysis

The first evaluation run using the custom LLM judge returned with an average of 3.6 points. For this run, 5 questions were only evaluated using the agent with the 120b LLM model integrated. 3 of the evaluation attempts about the cleanest lunch at Subway question were recorded as null due to "Request limit exceeded" error, which ended up resulting in an error in the LLM judge assessment. The same question's response was evaluated as 1 by the LLM judge due to the agent reaching maximum iterations. The agent correctly rejected the irrelevant question (Who won the NBA finals?), and the LLM judge evaluated this response as 5. The additive analysis response was evaluated as 5, while the Chick-fil-A Spicy Deluxe Sandwich analysis response was evaluated as 4, and the comparison response was evaluated as 3. The LLM judge did not provide any rationale for any of the responses. Overall, the LLM agent evaluated the questions in the evaluation dataset similarly to what was observed in the manual trace observation section, with the exception that one response did not reach the maximum iteration limit in this run. Our score was 3.2 for the 120b model.

For the human feedback section, due to SIMBA requiring a minimum of 10 traces, various brand new questions testing the different Unity Catalog functions were generated. These questions were also evaluated by the custom LLM judge, and human feedback was provided in the "Experiments" section of Databricks for every 10 traces.  

Key findings during human feedback:
- The agent correctly identified and rejected the irrelevant but nontrivial question. LLM evaluated this with a score of 5, and the human feedback was also given as 5.
- The agent made too many calls for some of the queries, and the LLM has given 3 or 5, while human feedback was 3 for such acceptable responses.
- The ingredient search tool was used accurately by the agent and evaluated well by both the LLM judge and the human.
- Where the agent supplied an incorrect data type, no output was returned, hence the LLM judge could not provide an evaluation. Feedback score was 1 for this example.
- Most nutritious option-related questions are answered using the find_nutritious_meals tool. However, this tool does not provide additive analysis, and the agent is not retrieving the ingredients from the web to search the additives against the FDA dataset. The LLM judge evaluated this response as excellent (5), while human feedback was acceptable (3). The instruction to do additive analysis was stated in the agent's prompt when the agent was created.
- Additive analysis response was acceptable (3), however, some additives are found as an acronym on web search and the LLM is not always able to use expansion for the acronyms. An addiitonal web search or agent instruction will be needed to address this problem. The LLM judge graded this response as excellent (5).
- For another additive safety question (Red 40), the agent used the single additive check tool, but, when the tool did not return anything, the agent fabricated the answer. The human feedback was 1, while the LLM judge feedback was 2 for this example.
- The LLM judge evaluated the responses as 2, where the mximum iteration limit was reached, while the human feedback was 1 for these kind of scenarios.
- For meal comparison query, the multi-step web searches returned more additives than the agent checked against FDA dataset using main__default__analyze_ingredients_batch tool. Hence, the human feedback was 2 due to the response not being completely wrong. The LLM score was 3.


###Optimization Analysis

Although we tried to evaluate the agent with different scores than the LLM judge, the scores were mostly aligned with the LLM judge's scores. SIMBA optimizer was ran multiple times with max_demos 1 and 2, and the feedback was changed to be more strict to achive an optimized LLM judge. However, the aligned judge was returned the same as the original judge every time. The original judge was created to be strict and it covered multiple areas. For future work, more evaluation traces can be created along with different optimization methods to achieve an optimized LLM judge.

###Conclusion

Using the 120b model agent, the first manual human analysis (on 5 traces) resulted in an average score of 3.2 out of 5, while the second run (10 traces) with human feedback returned an average score of 2.7. On the second evaluation run, the human feedback was forced to be strict to achieve an optimized custom judge. Considering these scores, the total average of all these 15 traces is approximately 3 out of 5, which is at an acceptable level for this limited agent system.

###Limitations and Future Work

The FDA and restaurant dataset used in this project were compressed and had limited information. More reliable datasets can be used in future work when the Databricks environment is available to support larger datasets. Some functions return errors depending on the LLM model input. This can be addressed by modifying the functions, improving the agent prompt, or using a more capable LLM model within the agent.

Future work may focus on user interface development, leveraging more advanced models to power the agent, integrating reliable data sources into the platform, and applying prompt optimization methods such as GEPA to improve both the agent prompt and the LLM-based judge instructions.